# Seed-based functional connectivity

---

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO/blob/main/Borriero_2_seed_based_functional_connectivity.ipynb)

In [ ]:
from google.colab import auth; auth.authenticate_user()
from google.colab import drive; drive.mount('/content/drive')

# Import

In [ ]:
# Install the required libraries
! pip install nibabel nilearn nitime statsmodels

In [3]:
import nilearn as nil
import nibabel as nib
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from nilearn import plotting, surface, datasets
from scipy.stats import pearsonr, spearmanr, ttest_ind
from tqdm import tqdm # For having progressbar during loops
from nitime.timeseries import TimeSeries
from nitime.analysis import FilterAnalyzer
from statsmodels.stats.multitest import multipletests

# Download some files

In [ ]:
# A wrapper method to download files

import os, subprocess, shlex
import os
import subprocess

def download_from_drive(file_keyword, base_folder="."):
    file_name, file_ID = files_dict[file_keyword]
    DEST_PATH = os.path.join(base_folder, file_name)
    if not os.path.exists(DEST_PATH):
        URL = f"https://drive.usercontent.google.com/download?id={file_ID}&confirm=t"
        cmd = ["wget", "-q", "--show-progress", "--progress=bar:force:noscroll",
            "--no-check-certificate", "-O", DEST_PATH, URL]
        ret = subprocess.call(cmd)  # shell=False by default
        if ret != 0:
            raise RuntimeError("Download failed. Check sharing settings or FILE_ID.")
        else:
            print("Downloaded " + file_name + "!")
    else:
        print(file_name + " already present:", DEST_PATH)   

In [ ]:
files_dict = {'glasser_labels' : ('glasser_labels.csv','12LumqrRkp1rUAJGg8h3tOIwWubhyyoqY'),
              'glasser_atlas':   ('glasser_atlas.nii.gz','1GeB5pi25SRnyit8s5siDUCFFmJfsB3Yw'),
              'schaefer_labels': ('schaefer_labels.csv', '1Lgh4X9hud2xXGxYEWADFxfMe81P_tTfS'),
              'schaefer_atlas':  ('schaefer_atlas.nii.gz', '1acSTf6m9TyJxm2ruTcwd9eC8PNGfG1PX'),
              'subj_01_rest':  ('subj_01_rest.nii.gz', '1XXhVrMymlaBE0ekbgbHoH8fVYteWFGIa'),
              'subj_03_rest':  ('subj_03_rest.nii.gz', '1RziRBrPUFemlHfjPMoeJX26CbHhqlKM4'),
              'subj_04_rest':  ('subj_04_rest.nii.gz', '10thmeWWgk31FaBk0Qi0n_rtOef5YVAjM'),
              'subj_05_rest':  ('subj_05_rest.nii.gz', '1-_ifL8yQWAoUiU7v7niz6O9YbVGyj4xn'),
              'subj_06_rest':  ('subj_06_rest.nii.gz', '1iE-5pfybf8imraDfJvP6A-XuXIyHVHQD'),
              'subj_01_sleep':  ('subj_01_sleep.nii.gz', '1zAU1sI9jS0wzGp_raNUemxUYPilLIZJH'),
              'subj_03_sleep':  ('subj_03_sleep.nii.gz', '1C9pPi7PwTcdg4Nfz1xVVCFToghRTxSQQ'),
              'subj_04_sleep':  ('subj_04_sleep.nii.gz', '1fn6xCluS7lEasPKgHafThwvhA11Mo0A5'),
              'subj_05_sleep':  ('subj_05_sleep.nii.gz', '1DkWrsAa3-jLtVggLIujtxNfqhgwezhea'),
              'subj_06_sleep':  ('subj_06_sleep.nii.gz', '1PvG2KQMF40QnmNs6Z5kMx8abKk_tBH3C'),
              'all_subj_mask':  ('000_all_subj_mask.nii.gz', '1azzh7yXC4uG8clHv4Zm7sBZgta0dTMDf')}

In [ ]:
for key in files_dict.keys():
  download_from_drive(key)

# Load an example  subject

In [ ]:
main_path = '/content/'
file_name = 'subj_01_rest.nii.gz'

sub_ = nib.load(main_path+file_name) # Load the nii.gz file of the subject
sub_header = sub_.header # Subject's header
sub_affine = sub_.affine # Subject's affine
sub = sub_.get_fdata() # Get the numpy version of the nii.gz file
print(sub.shape) # x*y*z*time

brain_mask = np.zeros((60, 72, 60))
brain_mask[np.where(sub.mean(axis=-1))!=0] =1
n_vox_brain_mask = int(np.sum(brain_mask))

# Load the mask

In [ ]:
main_path = '/content/'
file_name = 'all_subj_mask.nii.gz' 
mask_ = nib.load(main_path+file_name)
mask = mask_.get_fdata()
n_vox_mask = int(np.sum(mask))

# Load the atlas
In order to define regions of the brain, we need an atlas

In [ ]:
main_path = '/content/'
file_name = 'glasser_atlas.nii.gz'
file_name_labels = 'glasser_labels.csv'
path = main_path+file_name
path_lab = main_path+file_name_labels

atlas_ = nib.load(path)
atlas = atlas_.get_fdata()
atlas_header = atlas_.header
atlas_affine = atlas_.affine
atlas_labels = pd.read_csv(path_lab, delimiter=',')
print(atlas.shape)

# Define a mask from the atlas
mask = np.zeros(atlas.shape)
mask[np.where(atlas!=0)]=1
n_vox_mask = int(np.sum(mask))

In [ ]:
# Plot the atlas
plotting.plot_roi(atlas_, title='glasser', display_mode='ortho')

In [ ]:
atlas_labels

In [ ]:
# Subdivision in cortices of the glasser atlas
# network_list = atlas_labels['ROI_network'].unique().tolist()
network_list = atlas_labels['cortex'].unique().tolist()
network_list

# Exploring the functional connectivity of the motor cortex
Chose the motor cortex of one of the two emisphere as the seed area for the seed-based functional connectivity analysis

### Define the seed area

In [ ]:
hemi = 'LH'
seed_areas = atlas_labels.loc[atlas_labels['regionLongName']=='Primary_Motor_Cortex_L']
seed_idxs = seed_areas['regionID'].values # Indexes of the seed areas
seed_areas

# seed_areas = seed_areas.loc[seed_areas['Hemisphere']==hemi]
# seed_idxs = seed_areas['ROI_label'].values # Indexes of the seed areas
# seed_areas

In [ ]:
# Create a brain mask of our seed region
seeds_mask = np.zeros(atlas.shape)
for seed in seed_idxs:
    seeds_mask += (atlas==seed)

In [ ]:
## Plot the seed region

# Plot over the volume of the brain
seeds_img = nib.Nifti1Image(seeds_mask.astype(np.int16), atlas_affine) # Create a new image for the selected region
plotting.plot_roi(seeds_img, display_mode='ortho', title='Left Primary Motor Cortex')
plotting.show()

# Plot over the surface of the brain ##
fsaverage = datasets.fetch_surf_fsaverage()
texture_left = surface.vol_to_surf(seeds_img, fsaverage.pial_left)
texture_left[np.where(texture_left!=0)] = 1

fig, axs = plt.subplots(1,2, subplot_kw={'projection': '3d'})
# left hemisphere
plotting.plot_surf_roi(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       colorbar=False, view='lateral',axes=axs[0])
plotting.plot_surf_roi(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       colorbar=False, view='medial', axes=axs[1])
fig.suptitle('Left Primary Motor Cortex', x=0.51, y=0.7)

plotting.show()

# Compute the whole brain functional connectivity map for the seed region

## At first, compute the average time series of the seed region

In [ ]:
# Apply the seed's mask to the whole brain, then average the time series of the voxels within the seed, in order
# to obtain the time series off the whole seed
sub = sub
seed_voxels = sub[seeds_mask==1]
print(seed_voxels.shape)
seed_ts = seed_voxels.mean(axis=0)
print(seed_ts.shape)

We are assuming an isotropy of the signal inside the seed region...

In [ ]:
# Let's check the distribution of the activity values of the voxels inside the seed region
t = 10 # choose a time point
fig = plt.figure(figsize=(5,3))
plt.hist(seed_voxels[:,t], bins=50);
plt.title('t='+str(t));

In [ ]:
# Visualize the average time series of the seed region
fig = plt.figure(figsize=(6,3))
plt.plot(seed_ts);
plt.title("Average seed's time series")
# plt.xlim(0,20)

## Correlate the time series of the seed with all the other voxels of the brain

In [ ]:
sub = sub
# At first, choose the coorrelation measure
corr_measure = pearsonr
# corr_measure = spearmanr
fcmap = np.zeros(mask.shape) # Initialize the functional connectivity map
fcmap_p = np.zeros(mask.shape) # p-values related the correlation measure
mask_cord = np.where(mask==1) # Coordinates of all the voxel within the brain mask

# Loop over all the voxel of the brain
for v in tqdm(range(n_vox_mask)):
    voxel_ts = sub[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
    corr = corr_measure(seed_ts, voxel_ts)
    fcmap[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[0]
    fcmap_p[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[1]


In [ ]:
# Plot the FC maps
# threshold the Fc maps with the pvalues 
fcmap_thresholded = np.copy(fcmap)
fcmap_thresholded[np.where(fcmap_p>0.001/n_vox_mask)] = 0

fsaverage = datasets.fetch_surf_fsaverage()
fcmap_img = nib.Nifti1Image(fcmap_thresholded, sub_affine)
texture_left = surface.vol_to_surf(fcmap_img, fsaverage.pial_left)
texture_right = surface.vol_to_surf(fcmap_img, fsaverage.pial_right)

thresh = 0.2 # Threshold for the brain plots

fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
# Left hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                       title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
fig.suptitle('Left Primary Motor Cortex FC', x=0.51, y=1)
plt.tight_layout()
plotting.show()

## Try to clean the signal before the fc analysis

In [ ]:
# An example of filtering with a bandpass filter
sub_filtered = np.copy(sub)
TR = sub_.header.get_zooms()[3]
for v in tqdm(range(15,20)):
    voxel_ts = sub[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
    T = TimeSeries(voxel_ts.T, sampling_interval=TR)
    F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
    tmp_TC_filt = F.filtered_boxcar.data
    tmp_TC_filt = tmp_TC_filt.T
    sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt
    fig,axs = plt.subplots(1,2, figsize=(9,3))
    axs[0].plot(tmp_TC_filt, alpha=1)
    axs[0].set_title('filtered')
    axs[1].plot(voxel_ts, alpha=1)
    axs[1].set_title('unfiltered')


In [ ]:
# Filter an entire subject
sub_filtered = np.copy(sub)
TR = sub_.header.get_zooms()[3]
for v in tqdm(range(n_vox_mask)):
    voxel_ts = sub[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
    T = TimeSeries(voxel_ts.T, sampling_interval=TR)
    F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
    tmp_TC_filt = F.filtered_boxcar.data
    tmp_TC_filt = tmp_TC_filt.T
    sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt
#     plt.plot(tmp_TC_filt, label='filtered', alpha=0.75)
#     plt.plot(voxel_ts, label='original', alpha=0.75)
#     plt.legend()

In [ ]:
# Now perform the analysis with the filtered subject

# Extract the time series of the seed region
seed_voxels = sub_filtered[seeds_mask==1]
seed_ts = seed_voxels.mean(axis=0)

# At first, choose the coorrelation measure
corr_measure = pearsonr
# corr_measure = spearmanr

fcmap = np.zeros(mask.shape) # Initialize the functional connectivity map
fcmap_p = np.zeros(mask.shape) # p-values related the correlation measure
mask_cord = np.where(mask==1) # Coordinates of all the voxel within the brain mask

## Correlation ##
# Loop over all the voxel of the brain
for v in tqdm(range(n_vox_mask)):
    voxel_ts = sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
    corr = corr_measure(seed_ts, voxel_ts)
    fcmap[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[0]
    fcmap_p[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[1]

In [ ]:
plt.hist(fcmap[fcmap!=0]);

In [ ]:
# Plot the FC map
# threshold the FC map with the pvalues 
fcmap_thresholded = np.copy(fcmap)
fcmap_thresholded[np.where(fcmap_p>0.001/n_vox_mask)] = 0

fsaverage = datasets.fetch_surf_fsaverage()
fcmap_img = nib.Nifti1Image(fcmap_thresholded, sub_affine)
texture_left = surface.vol_to_surf(fcmap_img, fsaverage.pial_left)
texture_right = surface.vol_to_surf(fcmap_img, fsaverage.pial_right)

thresh = 0.5 # Threshold for the brain plots

fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
# Left hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                       title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
plt.tight_layout()
plotting.show()

# Compute the functional connectivity for many subjects

In [ ]:
# sub_list = ['01','03','04','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','22','23','24','25','26','27','28','29','30','31','32','33']
# sub_list = ['01','03','04','05','06','07','08','09','10','11']
sub_list = ['01','03','04','05','06']

# At first, choose the coorrelation measure
corr_measure = pearsonr
# corr_measure = spearmanr
mask_cord = np.where(mask==1) # Coordinates of all the voxel within the brain mask


fcmap_allsub = np.zeros(mask.shape)[np.newaxis,:,:,:]
fcmap_p_allsub = np.zeros(mask.shape)[np.newaxis,:,:,:]

# Loop over the subjects
main_path = '/content/'
for sub_i,sub in enumerate(sub_list): # Loop over the subjects    
    print(sub)
    file_name = 'subj_'+sub+'_rest.nii.gz'
    sub_ = nib.load(main_path+file_name) # Load the nii.gz file of the subject
    sub_header = sub_.header # Subject's header
    sub_affine = sub_.affine # Subject's affine
    sub = sub_.get_fdata() # Get the numpy version of the nii.gz file
    
    # Filter the signal
    sub_filtered = np.copy(sub)
    TR = sub_.header.get_zooms()[3]
    print('Filtering the signal...')
    for v in tqdm(range(n_vox_mask)):
        voxel_ts = sub[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
        T = TimeSeries(voxel_ts.T, sampling_interval=TR)
        F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
        tmp_TC_filt = F.filtered_boxcar.data
        tmp_TC_filt = tmp_TC_filt.T
        sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt

    # Compute the average time series of the seed region
    seed_voxels = sub_filtered[seeds_mask==1]
    seed_ts = seed_voxels.mean(axis=0)

    fcmap = np.zeros(mask.shape) # Initialize the functional connectivity map
    fcmap_p = np.zeros(mask.shape) # p-values related the correlation measure

    # Loop over all the voxel of the brain
    print('Computing the correlation...')
    for v in tqdm(range(n_vox_mask)):
        voxel_ts = sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
        corr = corr_measure(seed_ts, voxel_ts)
        fcmap[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[0]
        fcmap_p[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[1]
    
    fcmap_allsub = np.concatenate((fcmap_allsub, fcmap[np.newaxis,:,:,:]))
    fcmap_p_allsub = np.concatenate((fcmap_p_allsub, fcmap_p[np.newaxis,:,:,:]))

In [ ]:
# Plot the FC map

av_fcmap_allsub = fcmap_allsub.mean(axis=0)
av_fcmap_p_allsub = fcmap_p_allsub.mean(axis=0)
# # threshold the Fc maps with the pvalues 
# fcmap_thresholded = np.copy(av_fcmap_allsub)
# fcmap_thresholded[np.where(av_fcmap_p_allsub>0.05)] = 0

fsaverage = datasets.fetch_surf_fsaverage()
fcmap_img = nib.Nifti1Image(av_fcmap_allsub, sub_affine)
texture_left = surface.vol_to_surf(fcmap_img, fsaverage.pial_left)
texture_right = surface.vol_to_surf(fcmap_img, fsaverage.pial_right)

thresh = 0.4 # Threshold for the brain plots

fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(8,8))
# Left hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                       title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                            view='medial', colorbar=True, threshold=thresh, axes=axs[1][1])
plt.tight_layout()
plotting.show()

# Let's try to find the Default Mode Network

## Define the seed areas
We choose the medial frontal cortex, one of the main regions of the default mode network

In [ ]:
# seed_areas = atlas_labels.loc[atlas_labels['cortex']=='Somatosensory_and_Motor']
# seed_areas = atlas_labels.loc[atlas_labels['regionLongName']=='PreCuneus_Visual_Area_L']
seed_areas_names = ['Area_10r_L','Area_10v_L', 'Area_9_Middle_L']
seed_idxs = []
for s in seed_areas_names:
    seed_areas = atlas_labels.loc[atlas_labels['regionLongName']==s]
    seed_idx = seed_areas['regionID'].values # Indexes of the seed areas
    seed_idxs.append(seed_idx[0])

In [ ]:
# Make the brain mask of the seed
seeds_mask = np.zeros(atlas.shape)
for seed in seed_idxs:
    seeds_mask += (atlas==seed)

In [ ]:
### Plot the seed area ###
## Plot on the volume of the brain

# Create a new image for the selected region
seeds_img = nib.Nifti1Image(seeds_mask.astype(np.int16), atlas_affine)
# Plot the selected region
plotting.plot_roi(seeds_img, display_mode='ortho', title='Left medial frontal cortex')
plotting.show()

# Plot over the surface of the brain ##
fsaverage = datasets.fetch_surf_fsaverage()
texture_left = surface.vol_to_surf(seeds_img, fsaverage.pial_left)
texture_left[np.where(texture_left!=0)] = 1

fig, axs = plt.subplots(1,2, subplot_kw={'projection': '3d'})
# left hemisphere
plotting.plot_surf_roi(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       colorbar=False, view='lateral',axes=axs[0])
plotting.plot_surf_roi(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       colorbar=False, view='medial', axes=axs[1])
fig.suptitle('Left medial frontal cortex', x=0.51, y=0.7);

In [ ]:
### Signal filtering ###
sub_filtered = np.copy(sub)
TR = sub_.header.get_zooms()[3]
for v in tqdm(range(n_vox_mask)):
    voxel_ts = sub[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
    T = TimeSeries(voxel_ts.T, sampling_interval=TR)
    F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
    tmp_TC_filt = F.filtered_boxcar.data
    tmp_TC_filt = tmp_TC_filt.T
    sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt

In [ ]:
# Extract the time series of the seed region
seed_voxels = sub_filtered[seeds_mask==1]
seed_ts = seed_voxels.mean(axis=0)

# At first, choose the coorrelation measure
corr_measure = pearsonr
# corr_measure = spearmanr

# Initialize stuff
fcmap = np.zeros(mask.shape) # Initialize the functional connectivity map
fcmap_p = np.zeros(mask.shape) # p-values related the correlation measure
mask_cord = np.where(mask==1) # Coordinates of all the voxel within the brain mask

## Correlation ##
# Loop over all the voxel of the brain
for v in tqdm(range(n_vox_mask)):
    voxel_ts = sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
    corr = corr_measure(seed_ts, voxel_ts)
    fcmap[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[0]
    fcmap_p[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[1]

In [ ]:
# threshold the Fc maps with the pvalues 
fcmap_thresholded = np.copy(fcmap)
fcmap_thresholded[np.where(fcmap_p>0.01/n_vox_mask)] = 0

fsaverage = datasets.fetch_surf_fsaverage()
fcmap_img = nib.Nifti1Image(fcmap_thresholded, sub_affine)
texture_left = surface.vol_to_surf(fcmap_img, fsaverage.pial_left)
texture_right = surface.vol_to_surf(fcmap_img, fsaverage.pial_right)

thresh = 0.35 # Threshold for the brain plots

fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
# Left hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                       title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
plt.tight_layout()
plotting.show()

## Let's do the analysis for all the subjects

In [ ]:
main_path='/content/' # Path to the folder with the subjects
sub_list = ['01','03','04','05','06']

# At first, choose the coorrelation measure
corr_measure = pearsonr
# corr_measure = spearmanr
mask_cord = np.where(mask==1) # Coordinates of all the voxel within the brain mask


fcmap_allsub = np.zeros(mask.shape)[np.newaxis,:,:,:]
fcmap_p_allsub = np.zeros(mask.shape)[np.newaxis,:,:,:]

# Loop over the subjects
for sub in sub_list:
    sub_file = 'subj_'+sub+'_rest.nii.gz'
    print(sub_file)
    sub_ = nib.load(main_path+sub_file) # Load the nii.gz file of the subject
    sub_header = sub_.header # Subject's header
    sub_affine = sub_.affine # Subject's affine
    sub = sub_.get_fdata() # Get the numpy version of the nii.gz file
    
    # Filter the signal
    sub_filtered = np.copy(sub)
    TR = sub_.header.get_zooms()[3]
    print('Filtering the signal...')
    for v in tqdm(range(n_vox_mask)):
        voxel_ts = sub[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
        T = TimeSeries(voxel_ts.T, sampling_interval=TR)
        F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
        tmp_TC_filt = F.filtered_boxcar.data
        tmp_TC_filt = tmp_TC_filt.T
        sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt

    # Compute the average time series of the seed region
    seed_voxels = sub_filtered[seeds_mask==1]
    seed_ts = seed_voxels.mean(axis=0)

    fcmap = np.zeros(mask.shape) # Initialize the functional connectivity map
    fcmap_p = np.zeros(mask.shape) # p-values related the correlation measure

    # Loop over all the voxel of the brain
    print('Computing the correlation...')
    for v in tqdm(range(n_vox_mask)):
        voxel_ts = sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
        corr = corr_measure(seed_ts, voxel_ts)
        fcmap[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[0]
        fcmap_p[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[1]
    
    fcmap_allsub = np.concatenate((fcmap_allsub, fcmap[np.newaxis,:,:,:]))
    fcmap_p_allsub = np.concatenate((fcmap_p_allsub, fcmap_p[np.newaxis,:,:,:]))

In [ ]:
av_fcmap_allsub = fcmap_allsub.mean(axis=0)
av_fcmap_p_allsub = fcmap_p_allsub.mean(axis=0)
# # threshold the Fc maps with the pvalues 
# fcmap_thresholded = np.copy(av_fcmap_allsub)
# fcmap_thresholded[np.where(av_fcmap_p_allsub>0.05)] = 0

fsaverage = datasets.fetch_surf_fsaverage()
fcmap_img = nib.Nifti1Image(av_fcmap_allsub, sub_affine)
texture_left = surface.vol_to_surf(fcmap_img, fsaverage.pial_left)
texture_right = surface.vol_to_surf(fcmap_img, fsaverage.pial_right)

thresh = 0.45 # Threshold for the brain plots

fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
# Left hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                       title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
plt.tight_layout()
plotting.show()

# Compare the functional connectivity between rest and sleep

In [ ]:
# seed_areas = atlas_labels.loc[atlas_labels['cortex']=='Somatosensory_and_Motor']
# seed_areas = atlas_labels.loc[atlas_labels['regionLongName']=='PreCuneus_Visual_Area_L']
seed_areas_names = ['Area_10r_L','Area_10v_L', 'Area_9_Middle_L']
seed_idxs = []
for s in seed_areas_names:
    seed_areas = atlas_labels.loc[atlas_labels['regionLongName']==s]
    seed_idx = seed_areas['regionID'].values # Indexes of the seed areas
    seed_idxs.append(seed_idx[0])

# Make the brain mask of the seed
seeds_mask = np.zeros(atlas.shape)
for seed in seed_idxs:
    seeds_mask += (atlas==seed)

In [ ]:
### Compute the FC maps for the two conditions ###

main_path='/content/' # Path to the folder with the subjects
sub_list = ['01','03','04','05','06']
fc_all_cond = [] # where to store the fc of all the subjects of the two groups
for c, cond in enumerate(['rest','sleep']):

    # At first, choose the coorrelation measure
    corr_measure = pearsonr
    # corr_measure = spearmanr
    mask_cord = np.where(mask==1) # Coordinates of all the voxel within the brain mask


    fcmap_allsub = np.zeros(mask.shape)[np.newaxis,:,:,:]
    fcmap_p_allsub = np.zeros(mask.shape)[np.newaxis,:,:,:]

    # Loop over the subjects
    for sub in sub_list:
        print(sub)
        if cond=='rest':
            sub_file = 'subj_'+sub'_rest.nii.gz'
        else: # Sleep
            sub_file = 'subj_'+sub'_sleep.nii.gz'
        sub_ = nib.load(main_path+sub_file) # Load the nii.gz file of the subject
        sub_header = sub_.header # Subject's header
        sub_affine = sub_.affine # Subject's affine
        sub = sub_.get_fdata() # Get the numpy version of the nii.gz file
        
        # Filter the signal
        sub_filtered = np.copy(sub)
        TR = sub_.header.get_zooms()[3]
        print('Filtering the signal...')
        for v in tqdm(range(n_vox_mask)):
            voxel_ts = sub[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
            T = TimeSeries(voxel_ts.T, sampling_interval=TR)
            F = FilterAnalyzer(T, ub=0.08, lb=0.008) #0.08, 0.008
            tmp_TC_filt = F.filtered_boxcar.data
            tmp_TC_filt = tmp_TC_filt.T
            sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] = tmp_TC_filt

        # Compute the average time series of the seed region
        seed_voxels = sub_filtered[seeds_mask==1]
        seed_ts = seed_voxels.mean(axis=0)

        fcmap = np.zeros(mask.shape) # Initialize the functional connectivity map
        fcmap_p = np.zeros(mask.shape) # p-values related the correlation measure

        # Loop over all the voxel of the brain
        print('Computing the correlation...')
        for v in tqdm(range(n_vox_mask)):
            voxel_ts = sub_filtered[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v], :] # time series of the single voxel
            corr = corr_measure(seed_ts, voxel_ts)
            fcmap[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[0]
            fcmap_p[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = corr[1]
        
        fcmap_allsub = np.concatenate((fcmap_allsub, fcmap[np.newaxis,:,:,:]))
        fcmap_p_allsub = np.concatenate((fcmap_p_allsub, fcmap_p[np.newaxis,:,:,:]))
    fc_all_cond.append(fcmap_allsub)

In [ ]:
p_thresh = 0.05
corrected = False

cmap='coolwarm'    

t_brain = np.zeros(mask.shape)
p_brain_corr = np.zeros(mask.shape)
p_brain_uncorr = np.zeros(mask.shape)
for v in tqdm(range(n_vox_mask)): # Loop over all the voxels within the mask
    c1_voxel = fc_all_cond[0][:, mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] # time series of the single voxel
    c2_voxel = fc_all_cond[1][:, mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]]

    t, p = ttest_ind(c1_voxel, c2_voxel)
    t_brain[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = t
    if np.isnan(p):
        p_brain_uncorr[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = 1
    else:
        p_brain_uncorr[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = p
        

# Correction for multiple comparison
p_vals = p_brain_uncorr[mask==1]
p_vals_corr = multipletests(p_vals, alpha=0.05, method='fdr_bh')[1]
for v in tqdm(range(n_vox_mask)): # Loop over all the voxels within the mask
    p_brain_corr[mask_cord[0][v], mask_cord[1][v], mask_cord[2][v]] = p_vals_corr[v]


# Plot
if corrected:
    p_brain = np.copy(p_brain_corr) # Pre-Post average
else:
    p_brain = np.copy(p_brain_uncorr) # Pre-Post average

t_brain = np.copy(t_brain) 
p_brain[np.isnan(p_brain)]=0
t_brain[np.isnan(t_brain)]=0
t_brain[p_brain>p_thresh]=0

In [ ]:
# threshold the Fc maps with the pvalues 
fcmap_thresholded = np.copy(t_brain)
# fcmap_thresholded[np.where(fcmap_p>0.01/n_vox_mask)] = 0

fsaverage = datasets.fetch_surf_fsaverage()
fcmap_img = nib.Nifti1Image(fcmap_thresholded, sub_affine)
texture_left = surface.vol_to_surf(fcmap_img, fsaverage.pial_left)
texture_right = surface.vol_to_surf(fcmap_img, fsaverage.pial_right)

thresh = 0.1 # Threshold for the brain plots

fig, axs = plt.subplots(2,2, subplot_kw={'projection': '3d'}, figsize=(4,4))
# Left hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                       title='Left', view='lateral',colorbar=False, threshold=thresh, axes=axs[0][0])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                       title='Right', view='lateral', colorbar=False, threshold=thresh, axes=axs[0][1])
# Right hemisphere
plotting.plot_surf_stat_map(fsaverage.infl_left, texture_left, bg_map=fsaverage.sulc_left,hemi='left',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][0])
plotting.plot_surf_stat_map(fsaverage.infl_right, texture_right, bg_map=fsaverage.sulc_right,hemi='right',
                            view='medial', colorbar=False, threshold=thresh, axes=axs[1][1])
plt.tight_layout()
plotting.show()